# Silencio-2 — English TTS, tags, and voice clone on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/silencio-2-tts-voice-clone-ac39/colab/Silencio-2-tts-voice-clone.ipynb)

[`BlueRoseLabs/Silencio-2`](https://huggingface.co/BlueRoseLabs/Silencio-2) is a small **English** text-to-speech model: sound tags, emotion tags, streaming, and zero-shot cloning from a few seconds of clean audio. It is a fine-tune of [`kyutai/pocket-tts`](https://huggingface.co/kyutai/pocket-tts) 18M. **18.2M** parameters generate speech; **27.9M** including the Mimi encoder that cloning needs. All weights sit in one **112 MB** `model.safetensors`. Output is **24 kHz** mono. The card claims about 20× real-time on four Apple M4 CPU threads.

This is the free-Colab English TTS notebook with tags and cloning. The [Phonon 31M notebook](phonon-31m-en-streaming-tts.ipynb) is tinier English CPU TTS (no tags, no clone). [Maya 2 Native](Maya2-Native-indic-tts.ipynb) is 2.50B Indic + English. [Swar](Swar-voice-clone-tts.ipynb) is Nepali/English F5-TTS. [Drama 3](Drama-3-tts-voice-clone.ipynb) needs a Colab Pro A100.

Inference is the Hub-shipped package `silencio2` (`Silencio2TTS`). There is **no PyPI `silencio2` to install**, and you do **not** need `pocket-tts`. Snapshot the Hub repo, put that folder on `sys.path`, then `from silencio2 import Silencio2TTS`.

**Licence:** weights [CC-BY-4.0](https://creativecommons.org/licenses/by/4.0/) (same as Pocket TTS). Code in `silencio2/core` is MIT. See the repo [`NOTICE.md`](https://huggingface.co/BlueRoseLabs/Silencio-2/blob/main/NOTICE.md).

**Consent:** clone a voice only with its owner's permission. Do not impersonate people or deceive listeners. The clone demo below reuses a clip **this notebook just synthesised**. Label synthetic speech as synthetic where it could be mistaken for a person.

**Runtime:** *Runtime → Change runtime type → T4 GPU* if you have it. `Silencio2TTS.from_pretrained` uses CUDA when `torch.cuda.is_available()` and CPU otherwise. A T4 is fastest; **CPU also works** — the checkpoint is one 112 MB file. The package runs **float32**. T4 has no native bfloat16 and this path does not use FlashAttention-2. First run downloads ~112 MB of weights plus the bundled voice WAVs.

**Links:** [model card](https://huggingface.co/BlueRoseLabs/Silencio-2) · [Pocket TTS](https://huggingface.co/kyutai/pocket-tts) · [authors' Colab](https://colab.research.google.com/drive/1d6GSiRgXbYyBo91DCMrSr5y2LR_peJKy) · [announcement](https://x.com/julian_inada/status/2116987775467835840)


## 1. Install the `silencio2` dependencies (keep Colab's torch)

The card's install is `pip install -r requirements.txt` from a clone of [`BlueRoseLabs/Silencio-2`](https://huggingface.co/BlueRoseLabs/Silencio-2), then import `silencio2` from that folder. `requirements.txt` lists `torch>=2.5` and `numpy>=2` — **current Colab already has both**. This cell installs only the remaining libraries and does **not** replace Colab's CUDA torch.

Weights and the `silencio2/` package are **not** on PyPI. They come from the Hub snapshot in the load cell.


In [ ]:
import subprocess
import sys

print("python", sys.version.split()[0])

try:
    import torch

    print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
except ImportError:
    print("torch not imported yet")


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


# Card requirements.txt minus torch / numpy. Do not pip-install torch.
# scipy is imported by silencio2.core (wav I/O); Colab already has it.
pip_install(
    "sentencepiece>=0.2.1",
    "einops>=0.4",
    "soundfile>=0.12",
    "huggingface_hub>=0.13",
    "safetensors>=0.4",
    "pydantic>=2",
    "pyyaml",
    "scipy>=1.5",
)

print("deps ready — silencio2 itself is imported from the Hub snapshot below")


> Restart is usually unnecessary: this install does not upgrade `transformers` or replace torch. If a later cell cannot import `sentencepiece` or `einops`, use *Runtime → Restart session* and continue from the next cell. You do not need to rerun the install unless it failed.


## 2. Runtime check

Print `torch.cuda.is_available()`, the device name, and free VRAM. A T4 is preferred. CPU is fine if no GPU is attached — `from_pretrained` already picks `cuda` or `cpu`.


In [ ]:
import os
import shutil

import torch

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    DEVICE = "cuda"
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("device name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
    if major < 8:
        print("No native bfloat16 on this GPU; silencio2 runs float32 either way.")
else:
    DEVICE = "cpu"
    print("device name: CPU")
    print("VRAM free: n/a (CPU is fine — one 112 MB weight file)")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")
print("DEVICE", DEVICE)


## 3. Load Silencio-2 from a Hub snapshot

Same API as the [model card](https://huggingface.co/BlueRoseLabs/Silencio-2): snapshot the repo, put it on `sys.path`, then `Silencio2TTS.from_pretrained()`. Passing the local snapshot path avoids a second download. `device` is CUDA when a GPU is attached.

The first run downloads `model.safetensors` (~112 MB), `tokenizer.model`, `config.yaml`, `silencio2/`, and `voices/*.wav`.


In [ ]:
import sys
from pathlib import Path

from huggingface_hub import snapshot_download
from IPython.display import Audio, display

MODEL_ID = "BlueRoseLabs/Silencio-2"
SNAPSHOT = Path(snapshot_download(MODEL_ID))
if str(SNAPSHOT) not in sys.path:
    sys.path.insert(0, str(SNAPSHOT))

from silencio2 import EMOTIONS, SOUNDS, Silencio2TTS

tts = Silencio2TTS.from_pretrained(SNAPSHOT, device=DEVICE)

n_params = sum(p.numel() for p in tts.model.parameters())
print("repo:", MODEL_ID)
print("snapshot:", SNAPSHOT)
print("voices:", tts.voices)
print("sounds:", SOUNDS)
print("emotions:", EMOTIONS)
print(f"parameters: {n_params:,} ({n_params / 1e6:.1f}M)")
print("sample_rate:", tts.sample_rate)
print("device:", next(tts.model.parameters()).device)
if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    print(
        f"VRAM after load: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
        f"free / total {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB"
    )


## 4. Bundled voices

Eight voices ship in `voices/` (`tts.voices`): `1320`, `2094`, `260`, `3575`, `5639`, `6930`, `7127`, `7729`. Default is `260`. Same card line, one take each, `seed=7` so a rerun matches.


In [ ]:
LINE = "Hi, I'm Silencio."

for voice in tts.voices:
    path = f"silencio2-voice-{voice}.wav"
    tts.save(path, LINE, voice=voice, seed=7)
    print("voice", voice, "→", path)
    display(Audio(path))


## 5. Sound tags and emotion tags

Sounds are written where they happen: `[laugh]` `[sigh]` `[gasp]` `[cough]` `[clear_throat]`. Emotions apply to the text after them: `<happy>` `<sad>` `<angry>` `<fear>` `<surprised>` `<whisper>`. You can also pass `emotion="happy"` for the whole line. An unknown tag **raises** instead of being read aloud.

`<whisper>` is the least steady tag (card). It works best on short lines and varies by voice. `generate` / `save` raise a true whisper to speech level unless you pass `normalize=False`.


In [ ]:
TAG_DEMOS = [
    ("sound-sigh", "What a long day. [sigh]", {}),
    ("sound-laugh", "That was actually pretty funny. [laugh]", {}),
    ("sound-gasp", "Wait. [gasp] Did you hear that?", {}),
    ("sound-cough", "One second. [cough] As I was saying.", {}),
    ("sound-clear", "Ahem. [clear_throat] Ladies and gentlemen.", {}),
    ("emotion-happy", "We did it, we actually won!", {"emotion": "happy"}),
    ("emotion-whisper", "<whisper> Keep your voice down.", {}),
    ("emotion-mixed", "<whisper> Keep your voice down. <angry> I said stop!", {}),
]

for name, text, kw in TAG_DEMOS:
    path = f"silencio2-{name}.wav"
    tts.save(path, text, voice="260", seed=7, **kw)
    extra = f"  {kw}" if kw else ""
    print(f"{name}: {text}{extra}")
    display(Audio(path))


## 6. Voice clone from a short clip this notebook just made

`voice=` accepts a bundled id **or** a path to a few seconds of clean audio. There is no public third-party clip here — the reference is a take this notebook synthesised with bundled voice `7729` (the card's clone sample). That keeps the audio license-clean and avoids cloning a real person.

Only clone a voice you have permission to use.


In [ ]:
REF_TEXT = "Hi! This is my voice."
CLONE_TEXT = "Hello from a cloned take. Same speaker, new words."
ref_path = "silencio2-clone-ref.wav"
clone_path = "silencio2-clone.wav"

tts.save(ref_path, REF_TEXT, voice="7729", seed=7)
print("reference (bundled 7729):", REF_TEXT)
display(Audio(ref_path))

tts.save(clone_path, CLONE_TEXT, voice=ref_path, seed=7)
print("clone from", ref_path, "→", CLONE_TEXT)
display(Audio(clone_path))


## 7. Streaming chunks and real-time factor

`tts.stream` yields float32 chunks at `tts.sample_rate` as they are generated. It does **not** apply `generate()`'s loudness step. Time-to-first-audio (TTFA) is the wall time until the first chunk. Real-time factor here is `audio_seconds / wall_seconds` (higher is faster). The card's ~20× figure is an M4 CPU claim; a Colab T4 or CPU will print its own number.

The first use of a voice encodes the prompt. A one-word warmup is run first so TTFA is generation, not that encode.


In [ ]:
import time

import numpy as np

STREAM_TEXT = (
    "A longer text, played as it is generated. "
    "Silencio two writes twenty-four kilohertz speech one frame at a time, "
    "so the first audio can start before the rest of this paragraph is ready."
)

tts.generate("Ready.", voice="260", seed=7)

chunks = []
ttfa_s = None
t0 = time.perf_counter()
for chunk in tts.stream(STREAM_TEXT, voice="260", seed=7):
    if ttfa_s is None:
        ttfa_s = time.perf_counter() - t0
    chunks.append(chunk.reshape(-1).detach().cpu().float().numpy())
elapsed_s = time.perf_counter() - t0

if not chunks:
    raise SystemExit("The stream ended with no audio chunks.")

pcm = np.concatenate(chunks)
audio_s = len(pcm) / tts.sample_rate
first_ms = 1000.0 * len(chunks[0]) / tts.sample_rate
print(f"TTFA: {ttfa_s * 1000:.1f} ms")
print(f"first chunk: {len(chunks[0])} samples ({first_ms:.0f} ms at {tts.sample_rate} Hz)")
print(
    f"wall: {elapsed_s:.2f} s | audio: {audio_s:.2f} s | "
    f"realtime: {audio_s / elapsed_s:.1f}x | chunks: {len(chunks)}"
)
display(Audio(pcm, rate=tts.sample_rate))


## 8. Caveats and cleanup

- **Licence.** Weights are [CC-BY-4.0](https://creativecommons.org/licenses/by/4.0/). `silencio2/core` is MIT (Pocket TTS inference). Keep attribution to BlueRose Labs and Kyutai Pocket TTS.
- **English only.** Other languages are read with the wrong pronunciation.
- **Install.** Snapshot `BlueRoseLabs/Silencio-2` and import `silencio2` from that folder. Do not `pip install pocket-tts`. Do not look for `silencio2` on PyPI. Do not replace Colab's torch.
- **Tags.** Sounds go in `[square]` brackets; emotions in `<angle>` brackets. An unknown tag raises. Delivery varies by seed and voice — try a few seeds. `<whisper>` is the least steady and is best on short lines.
- **Loudness.** `generate` / `save` raise quiet takes (especially whisper) to speech level. `stream` leaves the raw level. Pass `normalize=False` to keep it.
- **Cloning.** A few seconds of clean audio. This notebook clones a clip it just synthesised. Do not upload a recording of a real person without consent.
- **Runtime.** T4 first; CPU also works. Float32, no FlashAttention-2. Weights are ~112 MB; peak VRAM on a short clip should stay well under 2 GB.
- **Not Phonon, not Maya 2, not Swar, not Drama 3.** Phonon is 64.5 MB English CPU TTS without tags. Maya 2 is 2.50B Indic streaming TTS. Swar is Nepali/English F5-TTS. Drama 3-int4 needs an A100.
- **Authors' Colab.** The card also links [an official Colab](https://colab.research.google.com/drive/1d6GSiRgXbYyBo91DCMrSr5y2LR_peJKy). This notebook is the `donvito/notebooks` T4 path with playback, clone, and streaming timing.

Free the GPU before you open another notebook:


In [ ]:
import gc

if torch.cuda.is_available():
    print(f"before cleanup: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

del tts
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()
    print(
        f"after cleanup: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
        f"free / total {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB"
    )
else:
    print("CPU runtime — nothing to empty on the GPU.")
